In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GoldLayer") \
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.1.0"
    ) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

26/05/09 18:10:37 WARN Utils: Your hostname, Saileshs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en0)
26/05/09 18:10:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/saileshpola/PycharmProjects/PythonProject/PysparkKafkaETE/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/saileshpola/.ivy2/cache
The jars for the packages stored in: /Users/saileshpola/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-230b1107-e6c3-429e-a736-dadcd0b320ab;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 1163ms :: artifacts dl 154ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default    

In [2]:
silver_df = spark.readStream.format("delta").load("../silverLayer/data/silver/trades")

In [3]:
from pyspark.sql.functions import col, window,sum,count,avg,expr, round
watermarked_df = silver_df.withWatermark("trade_timestamp","10 minutes")
watermarked_df = watermarked_df.select("trade_id","trader_id","quantity","price","trade_timestamp")

In [4]:
gold_df = (watermarked_df
           .groupBy(window("trade_timestamp","15 minutes"), col("trader_id"))
           .agg(
                sum("quantity").alias('total_quantity'),
                count("trade_id").alias('trade_count'),
                round(avg("price"), 2).alias('avg_price'),
                round(sum(expr("price * quantity")),2).alias("total_trade_value")
))

In [5]:
query = (gold_df.writeStream.format("delta").outputMode("append")
         .option("checkpointLocation", "checkpoints/gold/trader_metrics")
         .option("path", "data/gold/trader_metrics")
         .queryName("gold_trader_metrics")
         .start())
